<a href="https://colab.research.google.com/github/Moquiuti/python-inteligencia-artificial-aplicada/blob/main/desafio_final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q groq pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 3.3 MB/s eta 0:00:00


In [2]:
import os
import json
import time
import pandas as pd

from google.colab import userdata
from groq import Groq

In [3]:
try:
    os.environ["GROQ_API_KEY"] = userdata.get("CursoAluraPytonIaGenerative")
    client_groq = Groq()
    print("Cliente Groq configurado com sucesso.")
except Exception as e:
    print("Não foi possível configurar o cliente Groq.")
    print("Verifique se a chave 'CursoAluraPytonIaGenerative' foi salva corretamente nos Secrets do Colab.")
    print(f"Detalhe técnico: {e}")

Cliente Groq configurado com sucesso.


In [4]:
def demonstrar_tratamento_erros():
    print("=== Demonstração de tratamento de erros ===")

    # ValueError
    try:
        idade = int("vinte")
    except ValueError:
        print("ValueError: não foi possível converter o valor para número inteiro.")

    # TypeError
    try:
        resultado = "10" + 5
    except TypeError:
        print("TypeError: tipos incompatíveis em uma operação. Verifique se os dados precisam ser convertidos.")

    # ZeroDivisionError
    try:
        divisao = 10 / 0
    except ZeroDivisionError:
        print("ZeroDivisionError: não é possível dividir por zero.")

    # Exceção genérica + finally
    arquivo = None
    try:
        arquivo = open("arquivo_que_nao_existe.txt", "r", encoding="utf-8")
    except Exception as e:
        print("Ocorreu um erro não previsto ao tentar abrir um arquivo.")
        print(f"Detalhe técnico: {e}")
    finally:
        if arquivo:
            arquivo.close()
        print("Bloco finally executado: finalização concluída.\n")

demonstrar_tratamento_erros()

=== Demonstração de tratamento de erros ===
ValueError: não foi possível converter o valor para número inteiro.
TypeError: tipos incompatíveis em uma operação. Verifique se os dados precisam ser convertidos.
ZeroDivisionError: não é possível dividir por zero.
Ocorreu um erro não previsto ao tentar abrir um arquivo.
Detalhe técnico: [Errno 2] No such file or directory: 'arquivo_que_nao_existe.txt'
Bloco finally executado: finalização concluída.



In [5]:
def carregar_linhas_arquivo(nome_arquivo):
    arquivo = None
    try:
        arquivo = open(nome_arquivo, "r", encoding="utf-8")
        linhas = [linha.strip() for linha in arquivo if linha.strip()]
        print(f"Arquivo '{nome_arquivo}' carregado com sucesso.")
        print(f"Quantidade de linhas encontradas: {len(linhas)}")
        return linhas
    except FileNotFoundError:
        print(f"Arquivo '{nome_arquivo}' não encontrado. Verifique se ele foi enviado para o Colab.")
        return []
    except Exception as e:
        print("Ocorreu um erro ao ler o arquivo.")
        print(f"Detalhe técnico: {e}")
        return []
    finally:
        if arquivo:
            arquivo.close()
            print("Leitura finalizada e arquivo fechado corretamente.")

linhas_reviews = carregar_linhas_arquivo("Resenhas_App_ChatGPT.txt")

Arquivo 'Resenhas_App_ChatGPT.txt' carregado com sucesso.
Quantidade de linhas encontradas: 24
Leitura finalizada e arquivo fechado corretamente.


In [6]:
for i, linha in enumerate(linhas_reviews[:5], start=1):
    print(f"Linha {i}: {linha}")

Linha 1: 53409593$Safoan Riyad$J'aimais bien ChatGPT. Mais la derniÃ¨re mise Ã  jour a tout gÃ¢chÃ©. Elle a tout oubliÃ©.
Linha 2: 39485494$Habimana Therese$This app is very important but sometimes it gives lies
Linha 3: 4549594$Shahidatun jannat$Wonderful app..i just aastonished.. love this app..
Linha 4: 890535$Rayyan R$Anytime i try to get the app to work. It tells me "an error has occured" can this be fixed
Linha 5: 3590353$Nkanyi Cele$Acabo de instalar la aplicaciÃ³n desde Google Play Store, pero lamentablemente no se abre. Se queda dando vueltas.


In [7]:
def parse_linha_review(linha):
    try:
        partes = linha.split("$", 2)

        if len(partes) != 3:
            raise ValueError("A linha não está no formato esperado: id$usuario$resenha")

        return {
            "id_usuario": partes[0].strip(),
            "usuario": partes[1].strip(),
            "resenha_original": partes[2].strip()
        }

    except ValueError as e:
        print("Não foi possível interpretar uma das linhas do arquivo.")
        print(f"Motivo: {e}")
        return None

    except Exception as e:
        print("Ocorreu um erro inesperado ao processar uma linha.")
        print(f"Detalhe técnico: {e}")
        return None

In [8]:
reviews_estruturadas = []

for linha in linhas_reviews:
    item = parse_linha_review(linha)
    if item is not None:
        reviews_estruturadas.append(item)

print(f"Total de resenhas estruturadas: {len(reviews_estruturadas)}")
reviews_estruturadas[:3]

Total de resenhas estruturadas: 24


[{'id_usuario': '53409593',
  'usuario': 'Safoan Riyad',
  'resenha_original': "J'aimais bien ChatGPT. Mais la derniÃ¨re mise Ã  jour a tout gÃ¢chÃ©. Elle a tout oubliÃ©."},
 {'id_usuario': '39485494',
  'usuario': 'Habimana Therese',
  'resenha_original': 'This app is very important but sometimes it gives lies'},
 {'id_usuario': '4549594',
  'usuario': 'Shahidatun jannat',
  'resenha_original': 'Wonderful app..i just aastonished.. love this app..'}]

In [9]:
def analisar_review_com_llm(review_dict, max_tentativas=3):
    prompt = f"""
Você receberá uma resenha de aplicativo em qualquer idioma.

Retorne APENAS um JSON válido, sem bloco de código, sem explicações extras, no seguinte formato:

{{
  "usuario": "...",
  "resenha_original": "...",
  "traducao_pt_br": "...",
  "sentimento": "Positiva" ou "Negativa" ou "Neutra"
}}

Dados da resenha:
usuario: {review_dict["usuario"]}
resenha_original: {review_dict["resenha_original"]}
"""

    for tentativa in range(1, max_tentativas + 1):
        try:
            resposta = client_groq.chat.completions.create(
                model="openai/gpt-oss-20b",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Você é um assistente especialista em análise de resenhas. "
                            "Sempre devolva JSON válido, sem markdown, sem crases, sem comentários."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.2,
                reasoning_effort="low",
                max_completion_tokens=600
            )

            conteudo = resposta.choices[0].message.content

            if conteudo is None:
                raise ValueError("A resposta da IA veio vazia.")

            conteudo = conteudo.strip()

            # limpeza defensiva, caso a IA devolva bloco de código
            conteudo = conteudo.replace("```json", "").replace("```", "").strip()

            dados = json.loads(conteudo)

            return {
                "usuario": dados.get("usuario", review_dict["usuario"]),
                "resenha_original": dados.get("resenha_original", review_dict["resenha_original"]),
                "traducao_pt_br": dados.get("traducao_pt_br", ""),
                "sentimento": dados.get("sentimento", "Neutra")
            }

        except json.JSONDecodeError:
            print(f"Tentativa {tentativa}: a resposta não veio em JSON válido. Tentando novamente...")
            time.sleep(2)

        except ValueError as e:
            print(f"Tentativa {tentativa}: {e}")
            time.sleep(2)

        except Exception as e:
            print(f"Tentativa {tentativa}: não foi possível analisar a resenha neste momento.")
            print(f"Detalhe técnico: {e}")
            time.sleep(2)

    return {
        "usuario": review_dict["usuario"],
        "resenha_original": review_dict["resenha_original"],
        "traducao_pt_br": "Não foi possível gerar a tradução automaticamente.",
        "sentimento": "Neutra"
    }

In [10]:
teste = analisar_review_com_llm(reviews_estruturadas[0])
teste

{'usuario': 'Safoan Riyad',
 'resenha_original': "J'aimais bien ChatGPT. Mais la derniÃ¨re mise Ã  jour a tout gÃ¢chÃ©. Elle a tout oubliÃ©.",
 'traducao_pt_br': 'Eu gostava muito do ChatGPT. Mas a última atualização arruinou tudo. Ela esqueceu tudo.',
 'sentimento': 'Negativa'}

In [11]:
def processar_reviews_em_lote(lista_reviews):
    resultados = []

    for indice, review in enumerate(lista_reviews, start=1):
        print(f"Processando resenha {indice} de {len(lista_reviews)}...")
        resultado = analisar_review_com_llm(review)
        resultados.append(resultado)

    return resultados

reviews_processadas = processar_reviews_em_lote(reviews_estruturadas)

Processando resenha 1 de 24...
Processando resenha 2 de 24...
Processando resenha 3 de 24...
Processando resenha 4 de 24...
Processando resenha 5 de 24...
Processando resenha 6 de 24...
Processando resenha 7 de 24...
Processando resenha 8 de 24...
Processando resenha 9 de 24...
Processando resenha 10 de 24...
Processando resenha 11 de 24...
Processando resenha 12 de 24...
Processando resenha 13 de 24...
Processando resenha 14 de 24...
Processando resenha 15 de 24...
Processando resenha 16 de 24...
Processando resenha 17 de 24...
Processando resenha 18 de 24...
Processando resenha 19 de 24...
Processando resenha 20 de 24...
Processando resenha 21 de 24...
Processando resenha 22 de 24...
Processando resenha 23 de 24...
Processando resenha 24 de 24...


In [12]:
reviews_processadas[:3]

[{'usuario': 'Safoan Riyad',
  'resenha_original': "J'aimais bien ChatGPT. Mais la derniÃ¨re mise Ã  jour a tout gÃ¢chÃ©. Elle a tout oubliÃ©.",
  'traducao_pt_br': 'Eu gostava muito do ChatGPT. Mas a última atualização estragou tudo. Ela esqueceu tudo.',
  'sentimento': 'Negativa'},
 {'usuario': 'Habimana Therese',
  'resenha_original': 'This app is very important but sometimes it gives lies',
  'traducao_pt_br': 'Este aplicativo é muito importante, mas às vezes ele dá mentiras',
  'sentimento': 'Negativa'},
 {'usuario': 'Shahidatun jannat',
  'resenha_original': 'Wonderful app..i just aastonished.. love this app..',
  'traducao_pt_br': 'Aplicativo maravilhoso... fiquei maravilhado... amo este aplicativo...',
  'sentimento': 'Positiva'}]

In [13]:
df_reviews = pd.DataFrame(reviews_processadas)
display(df_reviews.head())

,usuario,resenha_original,traducao_pt_br,sentimento
0,Safoan Riyad,J'aimais bien ChatGPT. Mais la derniÃ¨re mise ...,Eu gostava muito do ChatGPT. Mas a última atua...,Negativa
1,Habimana Therese,This app is very important but sometimes it gi...,"Este aplicativo é muito importante, mas às vez...",Negativa
2,Shahidatun jannat,Wonderful app..i just aastonished.. love this ...,Aplicativo maravilhoso... fiquei maravilhado.....,Positiva
3,Rayyan R,Anytime i try to get the app to work. It tells...,"Sempre que tento fazer o aplicativo funcionar,...",Negativa
4,Nkanyi Cele,Acabo de instalar la aplicaciÃ³n desde Google ...,Acabei de instalar o aplicativo na Google Play...,Negativa


In [14]:
def consolidar_resultados(lista_reviews_processadas, separador="\n" + "="*80 + "\n"):
    try:
        contagem = {
            "Positiva": 0,
            "Negativa": 0,
            "Neutra": 0
        }

        partes_texto = []

        for item in lista_reviews_processadas:
            sentimento = item.get("sentimento", "Neutra")

            if sentimento in contagem:
                contagem[sentimento] += 1
            else:
                contagem["Neutra"] += 1

            trecho = (
                f"Usuário: {item.get('usuario', 'Desconhecido')}\n"
                f"Resenha original: {item.get('resenha_original', '')}\n"
                f"Tradução PT-BR: {item.get('traducao_pt_br', '')}\n"
                f"Sentimento: {item.get('sentimento', 'Neutra')}"
            )

            partes_texto.append(trecho)

        texto_unificado = separador.join(partes_texto)

        return contagem, texto_unificado

    except TypeError:
        print("Houve um problema de tipo de dado ao consolidar os resultados.")
        return None, ""

    except Exception as e:
        print("Ocorreu um erro inesperado ao consolidar os resultados.")
        print(f"Detalhe técnico: {e}")
        return None, ""

In [15]:
contagem_sentimentos, texto_final_unificado = consolidar_resultados(reviews_processadas)

print("Contagem final de sentimentos:")
print(contagem_sentimentos)

Contagem final de sentimentos:
{'Positiva': 9, 'Negativa': 15, 'Neutra': 0}


In [16]:
print(texto_final_unificado[:3000])

Usuário: Safoan Riyad
Resenha original: J'aimais bien ChatGPT. Mais la derniÃ¨re mise Ã  jour a tout gÃ¢chÃ©. Elle a tout oubliÃ©.
Tradução PT-BR: Eu gostava muito do ChatGPT. Mas a última atualização estragou tudo. Ela esqueceu tudo.
Sentimento: Negativa
Usuário: Habimana Therese
Resenha original: This app is very important but sometimes it gives lies
Tradução PT-BR: Este aplicativo é muito importante, mas às vezes ele dá mentiras
Sentimento: Negativa
Usuário: Shahidatun jannat
Resenha original: Wonderful app..i just aastonished.. love this app..
Tradução PT-BR: Aplicativo maravilhoso... fiquei maravilhado... amo este aplicativo...
Sentimento: Positiva
Usuário: Rayyan R
Resenha original: Anytime i try to get the app to work. It tells me "an error has occured" can this be fixed
Tradução PT-BR: Sempre que tento fazer o aplicativo funcionar, ele me diz "ocorreu um erro". Isso pode ser consertado?
Sentimento: Negativa
Usuário: Nkanyi Cele
Resenha original: Acabo de instalar la aplicaciÃ³n

In [17]:
try:
    with open("reviews_processadas.json", "w", encoding="utf-8") as arquivo_json:
        json.dump(reviews_processadas, arquivo_json, ensure_ascii=False, indent=2)
    print("Arquivo JSON salvo com sucesso.")
except Exception as e:
    print("Não foi possível salvar o arquivo JSON.")
    print(f"Detalhe técnico: {e}")

Arquivo JSON salvo com sucesso.


In [18]:
try:
    df_reviews.to_csv("reviews_processadas.csv", index=False, encoding="utf-8")
    print("Arquivo CSV salvo com sucesso.")
except Exception as e:
    print("Não foi possível salvar o arquivo CSV.")
    print(f"Detalhe técnico: {e}")

Arquivo CSV salvo com sucesso.


In [19]:
try:
    df_negativas = df_reviews[df_reviews["sentimento"] == "Negativa"]
    resenhas_negativas = df_negativas["traducao_pt_br"].dropna().tolist()

    texto_negativas_unificado = "\n#####\n".join(resenhas_negativas)

    print(f"Quantidade de resenhas negativas: {len(resenhas_negativas)}")
except Exception as e:
    print("Não foi possível preparar as resenhas negativas.")
    print(f"Detalhe técnico: {e}")
    texto_negativas_unificado = ""

Quantidade de resenhas negativas: 15


In [20]:
def extrair_categorias_reclamacoes(texto_negativas):
    if not texto_negativas.strip():
        return []

    prompt = f"""
A seguir há várias resenhas negativas já traduzidas para português.
Identifique as 5 categorias principais de reclamações.

Regras:
- responda apenas com 5 palavras
- cada categoria deve ter apenas uma palavra
- use minúsculas
- não use acentos
- separe por virgula

Resenhas:
{texto_negativas}
"""

    try:
        resposta = client_groq.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[
                {"role": "system", "content": "Responda de forma objetiva e no formato exato solicitado."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.2,
            reasoning_effort="low",
            max_completion_tokens=300
        )

        conteudo = resposta.choices[0].message.content

        if conteudo is None:
            return []

        categorias = [item.strip() for item in conteudo.strip().split(",") if item.strip()]
        return categorias

    except Exception as e:
        print("Não foi possível extrair as categorias das reclamações.")
        print(f"Detalhe técnico: {e}")
        return []

In [21]:
categorias_reclamacoes = extrair_categorias_reclamacoes(texto_negativas_unificado)
print(categorias_reclamacoes)

['bugs', 'performance', 'data', 'voice', 'pricing']
